In [2]:
!pip install -q datasets

from fastai.text.all import *
from datasets import load_dataset

In [3]:
raw_data = load_dataset("pminervini/HaluEval", "qa")

README.md: 0.00B [00:00, ?B/s]

qa/data-00000-of-00001.parquet:   0%|          | 0.00/3.75M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [4]:
print(raw_data)

DatasetDict({
    data: Dataset({
        features: ['knowledge', 'question', 'right_answer', 'hallucinated_answer'],
        num_rows: 10000
    })
})


In [5]:
example = raw_data['data'][0]
print("KNOWLEDGE:", example['knowledge'])
print("\nQUESTION:", example['question'])
print("\nRIGHT ANSWER:", example['right_answer'])
print("\nHALLUCINATED ANSWER:", example['hallucinated_answer'])



KNOWLEDGE: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.First for Women is a woman's magazine published by Bauer Media Group in the USA.

QUESTION: Which magazine was started first Arthur's Magazine or First for Women?

RIGHT ANSWER: Arthur's Magazine

HALLUCINATED ANSWER: First for Women was started first.


In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("OPENAI_API_KEY")

In [7]:
import json
import pandas as pd

#Testing the first 10 rows
df_all = pd.DataFrame(raw_data['data'])

all_tasks = []

for i, row in df_all.iterrows():
    lie_word_count = len(row['hallucinated_answer'].split())
    task = {
        "custom_id": f"test_{i}",

        "method": "POST",
        "url": "/v1/chat/completions",

        "body":{
            "model": "gpt-3.5-turbo-0125",
            "messages": [
                {"role": "system", "content": "You are a factual rewriter. Rewrite the statement to be 100% truthful based on context. You MUST use approximately {lie_word_count} words."},
                {"role": "user", "content": f"Context: {row['knowledge']}\nQuestion: {row['question']}"}
            ],

            "max_tokens": 60
        }
    }
    all_tasks.append(task)

with open("all_batch.jsonl", "w") as f:
    for task in all_tasks:
        f.write(json.dumps(task) + "\n")

print("File 'all_batch,jsonl' is ready on your Kaggle disk!")

File 'all_batch,jsonl' is ready on your Kaggle disk!


In [8]:
from openai import OpenAI
client = OpenAI(api_key=UserSecretsClient().get_secret("OPENAI_API_KEY"))

# This fetches the last 10 batches you ran
past_batches = client.batches.list(limit=10)

for batch in past_batches:
    print(f"ID: {batch.id} | Status: {batch.status} | Created At: {batch.created_at}")

ID: batch_696ba404b5708190b92b1ca0c6b96dce | Status: failed | Created At: 1768662020
ID: batch_696ba29f7d5c81909cb8c8efaecb1c1c | Status: completed | Created At: 1768661663
ID: batch_696a7f4b70b481909713bf67be14def2 | Status: completed | Created At: 1768587083
ID: batch_696a50f09fac8190953fb7132f22859c | Status: completed | Created At: 1768575216
ID: batch_696a4e2642d0819099898fe1614fc246 | Status: completed | Created At: 1768574502
ID: batch_696a4af77ac88190b5171094296bb1a0 | Status: completed | Created At: 1768573687
ID: batch_696970e1b8b881908e75f5b617e0e748 | Status: completed | Created At: 1768517857


In [9]:
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

client = OpenAI(api_key=UserSecretsClient().get_secret("OPENAI_API_KEY"))

all_file = client.files.create(
    file=open("all_batch.jsonl", "rb"),
    purpose="batch"
)

#Trigger the actual Job
all_job = client.batches.create(
    input_file_id=all_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

print(f"Batch Job created! ID: {all_job.id}")

Batch Job created! ID: batch_696d405f3dd48190b1ea1809982a1b95


In [10]:
job_status = client.batches.retrieve("batch_696a7f4b70b481909713bf67be14def2")
print(f"Status: {job_status.status}")

Status: completed


In [11]:
import time

# Paste the ID from your most recent run here
current_batch_id = "batch_696a7f4b70b481909713bf67be14def2"

while True:
    status_obj = client.batches.retrieve(current_batch_id)
    print(f"Status: {status_obj.status}")

    if status_obj.status == "completed":
        print("✅ Job finished! You can now run the retrieval code.")
        break
    elif status_obj.status in ["failed", "expired", "cancelled"]:
        print(f"❌ Job stopped with status: {status_obj.status}")
        break
    
    # Wait 30 seconds before checking again to avoid hitting rate limits
    time.sleep(30)

Status: completed
✅ Job finished! You can now run the retrieval code.


In [12]:
import json

# 1. Fetch the binary content from OpenAI using the ID from your completed job
# status_obj is the variable from your while-loop
file_response = client.files.content(status_obj.output_file_id)

# 2. Decode the binary bytes into a readable string
raw_content = file_response.content.decode("utf-8")

# 3. Parse the JSONL content into our dictionary
results_map = {}
lines = raw_content.strip().split('\n')

for line in lines:
    entry = json.loads(line)
    
    # This matches the "custom_id" you created earlier (e.g., "test_0")
    custom_id = entry['custom_id']
    
    # This extracts the actual text of the AI's truthful rewrite
    ai_answer = entry['response']['body']['choices'][0]['message']['content']
    
    # Map the ID to the answer for quick lookup
    results_map[custom_id] = ai_answer

print(f"Successfully retrieved {len(results_map)} AI Truths!")

Successfully retrieved 10000 AI Truths!


In [13]:
final_list = []

for i, row in df_all.iterrows():
    lie_text = f"{row['question']} [SEP] {row['hallucinated_answer']}"

    final_list.append({"text_input": lie_text, "label": 1})

    ai_truth = results_map.get(f"test_{i}")

    truth_text = f"{row['question']} [SEP] {ai_truth}"

    final_list.append({"text_input": truth_text, "label": 0})

In [14]:
df_train = pd.DataFrame(final_list)

df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_train.head())

                                                                                                                                                                                                                                                                                                                                                                                             text_input  \
0                                                                                                                                                                                                                                                                     Are Prevention and Drum! both American magazines? [SEP] No, Prevention is an American magazine while Drum! is a British magazine.   
1  The 2007 FIFA U-20 World Cup was the sixteenth edition of the FIFA U-20 World Cup (formerly called FIFA World Youth Championship), hosted by Canada held during which span of dates, Argentine player Sergio Ag

In [15]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [16]:
text_list = df_train['text_input'].tolist()
encodings = tokenizer.batch_encode_plus(
    text_list,
    add_special_tokens=True,
    max_length=256,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)

labels = torch.tensor(df_train['label'].values)

In [17]:
from torch.utils.data import TensorDataset, random_split, DataLoader, RandomSampler, SequentialSampler

dataset = TensorDataset(
    encodings['input_ids'],
    encodings['attention_mask'],
    labels
)

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [18]:
batch_size = 32

train_dataloader = DataLoader(
    train_dataset,
    sampler=RandomSampler(train_dataset),
    batch_size=batch_size
)

validation_dataloader = DataLoader(
    val_dataset,
    sampler=SequentialSampler(val_dataset),
    batch_size=batch_size
)

In [19]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [20]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

2026-01-18 20:20:16.828422: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768767617.004717      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768767617.058642      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768767617.460724      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768767617.460770      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768767617.460773      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

In [22]:
from transformers import get_linear_schedule_with_warmup

# 1. Ensure epochs is defined
epochs = 3 

# 2. Calculate the total number of training steps
# Total steps = [number of batches] * [number of epochs]
total_steps = len(train_dataloader) * epochs

# 3. Create the learning rate scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=0, # Default starting point
    num_training_steps=total_steps
)

In [23]:
for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    total_loss = 0

    for batch in train_dataloader: # Fixed 'bath' to 'batch'
        b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]
        model.zero_grad()

        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step() # --- ADD SCHEDULER STEP HERE ---

    print(f"Epoch {epoch+1} average training loss: {total_loss / len(train_dataloader)}")

    # --- ADD VALIDATION PHASE HERE (Inside the epoch loop) ---
    print("Running Validation...")
    model.eval() # Switch to evaluation mode
    val_accuracy = 0

    for batch in validation_dataloader:
        b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]
        
        with torch.no_grad(): # Don't calculate gradients to save memory
            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
        
        # We need to process the model's 'logits' to get the actual prediction
        logits = outputs.logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        
        # (We'll add accuracy calculation logic next!)
        tmp_eval_accuracy = flat_accuracy(logits, label_ids)
        
        # Accumulate the total accuracy
        val_accuracy += tmp_eval_accuracy

    # Report the final accuracy for this epoch
    avg_val_accuracy = val_accuracy / len(validation_dataloader)
    print(f"  Accuracy: {avg_val_accuracy:.2f}")

Epoch 1 average training loss: 0.37869513434478697
Running Validation...
  Accuracy: 0.86
Epoch 2 average training loss: 0.24980590624162402
Running Validation...
  Accuracy: 0.86
Epoch 3 average training loss: 0.1723398891629959
Running Validation...
  Accuracy: 0.86


In [24]:
import os
os.makedirs('lie_detector_app', exist_ok=True)

model.save_pretrained('./lie_detector_app')
tokenizer.save_pretrained('./lie_detector_app')

print("Model exported! You'll see a 'pytorch_model.bin' and config files in your folder.")

Model exported! You'll see a 'pytorch_model.bin' and config files in your folder.
